[目录](./table_of_contents.ipynb)

# 平滑

In [ ]:
%matplotlib inline

In [ ]:
#格式化书本
import book_format
book_format.set_style()

## 简介

考虑到未来的数据时，卡尔曼滤波器的性能并不理想。例如，假设我们正在追踪一架飞机，并且最新的测量结果偏离了当前轨迹，如下所示（为简单起见，我仅考虑1个维度）：

In [ ]:
import matplotlib.pyplot as plt

data = [10.1, 10.2, 9.8, 10.1, 10.2, 10.3, 
        10.1, 9.9, 10.2, 10.0, 9.9, 11.4]
plt.plot(data)
plt.xlabel('时间')
plt.ylabel('位置');

经过一段时间的近乎稳定状态后，我们有了一个非常大的变化。假设这个变化超出了飞机飞行包线的限制。尽管如此，卡尔曼滤波器根据当前的卡尔曼增益将新的测量值纳入了过滤器中。它不能拒绝噪声，因为这个测量值可能反映了一次转弯的开始。虽然我们很少会突然转弯，但是我们无法确定：

* 飞机之前开始转弯了，但是之前的测量值很嘈杂，没有显示出这个变化

* 飞机正在转弯，这个测量值非常嘈杂

* 这个测量值非常嘈杂，而且飞机没有转弯

* 飞机正在相反的方向转弯，这个测量值非常嘈杂

现在，假设以下测量值为：

   11.3 12.1 13.3 13.9 14.5 15.2

In [ ]:
data2 = [11.3, 12.1, 13.3, 13.9, 14.5, 15.2]
plt.plot(data + data2);

根据这些未来的测量数据，我们可以推断出飞机已经开始转弯了。

另一方面，假设以下是接下来的测量数据。

In [ ]:
data3 = [9.8, 10.2, 9.9, 10.1, 10.0, 10.3, 9.9, 10.1]
plt.plot(data + data3);

在这种情况下，我们得出结论，飞机没有转弯，而是测量值只是非常嘈杂。

## 平滑器的概述

卡尔曼滤波器是一种具有马尔可夫性的*递归*滤波器-它在步骤`k`的估计仅基于步骤`k-1`的估计和步骤`k`的测量。但是这意味着步骤`k-1`的估计基于步骤`k-2`，依此类推回到第一个时期。因此，步骤`k`的估计取决于所有以前的测量值，但程度不同。`k-1`有最大的影响，`k-2`有次大的影响，依此类推。

平滑滤波器将未来的测量值合并到步骤`k`的估计中。来自`k+1`的测量值将产生最大的影响，`k+2`的影响会更小，`k+3`的影响则更小，以此类推。

这个主题被称为*平滑*，但我认为这是一个误导性的名称。我可以通过低通滤波器传递数据来平滑上述数据。结果会很平滑，但不一定准确，因为低通滤波器会像删除噪声一样删除真实变化。相比之下，Kalman平滑器是*最优*的 - 它们结合所有可用的信息，使得数学上可实现最佳估计。

## 平滑器类型

有三类Kalman平滑器可以在这些情况下产生更好的跟踪效果。

* 固定间隔平滑

这是一种基于批处理的滤波器。该滤波器等待所有数据被收集后再进行估计。例如，您可能是一位科学家正在收集实验数据，并且不需要在实验完成之前知道结果。 固定间隔平滑器将收集所有数据，然后使用所有可用的先前和未来测量值在每个测量值处估计状态。 如果您可以以批处理模式运行卡尔曼滤波器，则始终建议使用这些滤波器，因为它们将比前几章中的递归滤波器提供更好的结果。

* 固定滞后平滑

固定滞后平滑器会使输出延迟。假设我们选择了4个步长的滞后。滤波器将输入前3个测量值，但不会输出滤波结果。当第4个测量值进入滤波器时，滤波器将产生第1个测量值的输出，考虑1到4个测量值。当第5个测量值进入时，滤波器将产生第2个测量值的结果，考虑2到5个测量值。当你需要最近的数据但又可以承受一点滞后时，这非常有用。例如，也许你正在使用机器视觉监控制造过程。如果你可以承受几秒钟的估计延迟，固定滞后平滑器将允许你产生非常准确和平滑的结果。

* 固定点平滑器

固定点滤波器的操作和普通的卡尔曼滤波器相同，但是它还会产生一个在某个固定时间点 $j$ 的状态估计。在时间 $k$ 达到 $j$ 之前，滤波器的操作就像普通滤波器一样。一旦 $k>j$，滤波器就会估计 $x_k$，然后使用 $j\dots k$ 之间的所有测量更新其对 $x_j$ 的估计。这可以用于估计系统的初始参数，或者用于产生在特定时间发生的事件的最佳估计。例如，您可能有一个机器人在时间 $j$ 拍摄了一张照片。您可以使用固定点平滑器来获取相机在时间 $j$ 的最佳可能姿态信息，同时机器人继续移动。

## 滤波器的选择

这些滤波器的选择取决于你的需求以及你可以拨出多少内存和处理时间。固定点平滑需要存储所有的测量值，而且计算成本非常高，因为输出对于每个测量值的时间步骤都要重新计算。另一方面，该滤波器确实为当前测量值产生了不错的输出，因此该滤波器可以用于实时应用。

固定滞后平滑只需要你存储一个数据窗口，处理要求较小，因为每个新的测量值只需要处理该窗口。缺点是滤波器的输出总是滞后于输入，平滑效果也没有固定间隔平滑时那么明显。

固定间隔平滑以最大限度地平滑输出为代价，需要批量处理。大多数算法使用某种前向/后向算法，其速度只比递归卡尔曼滤波器慢两倍。

## 固定间隔平滑

文献中提供了许多固定滞后平滑器。我选择实现Rauch、Tung和Striebel发明的平滑器，因为它易于实现且计算效率高。在实际应用中，我也经常看到使用这个平滑器。这个平滑器通常被称为RTS平滑器。

RTS平滑器的推导需要数页密密麻麻的数学公式。我不想让你们承受这个痛苦。相反，我将简要介绍算法和公式，然后直接进行平滑器的实现和演示。

RTS平滑器的工作方式是首先以批处理模式运行Kalman滤波器，计算每个步骤的滤波器输出。给定每个测量的滤波器输出以及对应于每个输出的协方差矩阵，RTS向后运行数据，将其对未来的了解纳入到过去的测量中。当它到达第一个测量时，它就完成了，滤波器的输出以最优形式将所有信息合并在一起。

RTS平滑器的方程非常直观且易于实现。这个推导是针对线性卡尔曼滤波器的。类似的推导也适用于EKF和UKF。这些步骤是在批处理输出上执行的，从最近的时间往回推到第一个估计值。每次迭代都将未来的知识合并到状态估计中。由于状态估计已经包含了所有过去的测量，结果将是每个估计值都包含了过去和未来的所有测量知识。在这里，非常重要的是区分过去、现在和未来，因此我使用下标来表示数据是否来自未来。

In [ ]:
预测步骤

$$\begin{aligned}
\mathbf{P} &= \mathbf{FP}_k\mathbf{F}^\mathsf{T} + \mathbf{Q }
\end{aligned}$$

更新步骤

$$\begin{aligned}
\mathbf{K}_k &= \mathbf{P}_k\mathbf{F}^\mathsf{T}\mathbf{P}^{-1} \\
\mathbf{x}_k &= \mathbf{x}_k + \mathbf{K}_k(\mathbf{x}_{k+1} - \mathbf{Fx}_k) \\
\mathbf{P}_k &= \mathbf{P}_k + \mathbf{K}_k(\mathbf{P}_{k+1} - \mathbf{P})\mathbf{K}_k^\mathsf{T}
\end{aligned}$$

像往常一样，实现最困难的部分是正确地处理下标。没有注释或错误检查的基本实现如下：

In [ ]:
def rts_smoother(Xs, Ps, F, Q):
    n, dim_x, _ = Xs.shape

    # 平滑器增益
    K = zeros((n,dim_x, dim_x))
    x, P, Pp = Xs.copy(), Ps.copy(), Ps.copy

    for k in range(n-2,-1,-1):
        Pp[k] = F @ P[k] @ F.T + Q # 预测协方差

In [ ]:
K[k]  = P[k] @ F.T @ inv(Pp[k])
        x[k] += K[k] @ (x[k+1] - (F @ x[k]))     
        P[k] += K[k] @ (P[k+1] - Pp[k]) @ K[k].T
    return (x, P, K, Pp)

这个实现与FilterPy中提供的实现相似。它假定卡尔曼滤波器在批处理模式下外部运行，并通过 `Xs` 和 `Ps` 变量传递状态和协方差的结果。

以下是一个例子。

In [ ]:
import numpy as np
from numpy import random
from numpy.random import randn
import matplotlib.pyplot as plt
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise
import kf_book.book_plots as bp

def plot_rts(noise, Q=0.001, show_velocity=False):
    random.seed(123)
    fk = KalmanFilter(dim_x=2, dim_z=1)

    fk.x = np.array([0., 1.])      # 状态 (x 和 dx)

    fk.F = np.array([[1., 1.],
                     [0., 1.]])    # 状态转移矩阵

In [ ]:
fk.H = np.array([[1., 0.]])    # 测量函数
    fk.P*= 10.                     # 协方差矩阵
    fk.R = noise                   # 状态不确定性
    fk.Q = Q_discrete_white_noise(dim=2, dt=1., var=Q)  # 过程不确定性

    # 创建带有噪声的数据
    zs = np.asarray([t + randn()*noise for t in range (40)])

    # 使用卡尔曼滤波器过滤数据，然后对其进行平滑处理
    mu, cov, _, _ = fk.batch_filter(zs)
    M, P, C, _ = fk.rts_smoother(mu, cov)

    # 绘制数据
    if show_velocity:
        index = 1
        print('gu')
    else:
        index = 0
    if not show_velocity:
        bp.plot_measurements(zs, lw=1)
    plt.plot(M[:, index], c='b', label='RTS')
    plt.plot(mu[:, index], c='g', ls='--', label='KF输出')
    if not show_velocity:
        N = len(zs)
        plt.plot([0, N], [0, N], 'k', lw=2, label='轨迹') 
    plt.legend(loc=4)
    plt.show()
    
plot_rts(7.)

我已经将大量噪声注入信号，以便您可以在图表中将RTS输出与理想输出区分开。在上面的图表中，我们可以看到Kalman滤波器以绿色虚线绘制，与输入相比相当平滑，但当连续几个测量偏向线路的一侧时，它仍会偏离理想线路。相比之下，RTS输出既非常平滑又非常接近理想输出。

通过可能更为合理的噪声量，我们可以看到RTS输出几乎位于理想输出上。Kalman滤波器的输出要好得多，但仍会产生更大的变化。

In [ ]:
plot_rts(noise=1.)

然而，我们必须明白这种平滑是建立在系统模型的基础之上的。我们已经告诉了滤波器，我们要跟踪的对象遵循一个具有非常低的过程误差的恒定速度模型。当滤波器“展望”未来时，它会发现未来的行为与恒定速度非常接近，因此能够排除信号中的大部分噪音。假设我们的系统有很多过程噪声。例如，如果我们在多风区跟踪一架小型飞机，它的速度会经常变化，滤波器将难以区分由于风引起的噪音和异常运动。我们可以在下一个图表中看到这一点。

In [ ]:
plot_rts(noise=7., Q=.1)

这强调了这些滤波器并没有在口语意义上对数据进行“平滑”。滤波器是基于先前的测量、未来的测量以及您告诉它的系统行为和系统及测量中的噪声进行最优估计。

让我们来看看卡尔曼滤波器和 RTS 平滑器的速度估计。

In [ ]:
plot_rts(7.,show_velocity=True)

速度的改进更加显著，因为它是一个隐藏变量。

## 固定滞后平滑

如果您可以批处理运行，则应始终选择 RTS 平滑算法，因为它将所有可用数据合并到每个估计中。并非所有问题都允许您这样做，但您可能仍然对接收以前估计的平滑值感兴趣。下面的数字线说明了这个概念。

In [ ]:
from kf_book.book_plots import figsize
from kf_book.smoothing_internal import *

with figsize(y=2):
    show_fixed_lag_numberline()

在第$k$步，我们可以使用正常的卡尔曼滤波器方程来估计$x_k$。然而，我们可以通过使用为$x_k$接收到的测量结果来更好地估计$x_{k-1}$。同样，我们可以通过使用为$x_{k-1}$和$x_{k}$接收到的测量结果来更好地估计$x_{k-2}$。我们可以将此计算向后扩展到任意$N$步。

这个数学推导超出了本书的范围；如果您感兴趣，丹·西蒙的《最优状态估计》[2]有很好的阐述。这个想法的本质是，我们不是拥有一个状态向量 $\mathbf{x}$，而是一个包含

$$\mathbf{x} = \begin{bmatrix}\mathbf{x}_k \\ \mathbf{x}_{k-1} \\ \vdots\\ \mathbf{x}_{k-N+1}\end{bmatrix}$$

的扩展状态。

这将产生一个非常大的协方差矩阵，其中包含不同步骤之间状态之间的协方差。FilterPy的类 `FixedLagSmoother` 为您处理所有这些计算，包括增广矩阵的创建。您所需要做的就像使用 `KalmanFilter` 类一样将其组合起来，然后调用 `smooth()`，它实现了算法的预测和更新步骤。

每次调用 `smooth` 计算当前测量的估计值，但它还会返回并调整前 N-1 个点。平滑后的值包含在列表 `FixedLagSmoother.xSmooth` 中。如果您使用 `FixedLagSmoother.x`，则会得到最近的估计值，但它未经平滑处理，并且与标准 Kalman 滤波器输出没有区别。

In [ ]:
from filterpy.kalman import FixedLagSmoother, KalmanFilter
import numpy.random as random

fls = FixedLagSmoother(dim_x=2, dim_z=1, N=8)

In [ ]:
fls.x = np.array([0., .5])
fls.F = np.array([[1.,1.],
                  [0.,1.]])

fls.H = np.array([[1.,0.]])
fls.P *= 200
fls.R *= 5.
fls.Q *= 0.001

kf = KalmanFilter(dim_x=2, dim_z=1)
kf.x = np.array([0., .5])
kf.F = np.array([[1.,1.],
                 [0.,1.]])
kf.H = np.array([[1.,0.]])
kf.P *= 200
kf.R *= 5.
kf.Q = Q_discrete_white_noise(dim=2, dt=1., var=0.001)

N = 4 # 滞后大小

nom =  np.array([t/2. for t in range (0, 40)])
zs = np.array([t + random.randn()*5.1 for t in nom])

for z in zs:
    fls.smooth(z)
    
kf_x, _, _, _ = kf.batch_filter(zs)
x_smooth = np.array(fls.xSmooth)[:, 0]


fls_res = abs(x_smooth - nom)
kf_res = abs(kf_x[:, 0] - nom)

plt.plot(zs,'o', alpha=0.5, label='zs')
plt.plot(x_smooth, label='FLS')
plt.plot(kf_x[:, 0], label='KF', ls='--')
plt.legend(loc=4)

print(f'固定滞后标准差: {np.mean(fls_res):.3f}')
print(f'卡尔曼标准差: {np.mean(kf_res):.3f}')

这里我设置了 `N=8`，这意味着我们将把8个未来的测量值纳入我们的估计中。一旦滤波器收敛，这将为我们提供非常平滑的估计，但计算量大约是标准卡尔曼滤波器的8倍。请随意尝试更大或更小的 `N` 值。我随便选择了8，没有任何理论上的考虑。

## 参考文献

- [1] H. Rauch, F. Tung, and C. Striebel. "Maximum likelihood estimates of linear dynamic systems," *AIAA Journal*, **3**(8), pp. 1445-1450 (August 1965). http://arc.aiaa.org/doi/abs/10.2514/3.3166

- [2] Dan Simon. "Optimal State Estimation," John Wiley & Sons, 2006.